# ECABSD V3 — Training Notebook
**Architecture:** 6-layer GATv2 + Cross-Attention + Global Fusion  
**Loss:** Combined Focal + Soft-Dice  
**Scheduler:** Linear Warmup → CosineAnnealingLR  
**Saves:** `checkpoints/best_model_v3.pt`

> Run cells top-to-bottom. Each cell prints status. GPU required.

In [ ]:
# ============================================================
# CELL 1: Wipe old code and clone clean repo
# ============================================================
import os, shutil

WORK = '/kaggle/working'
REPO = 'https://github.com/VigneshReddyKura/ecabsd.git'
DEST = f'{WORK}/ecabsd'

os.chdir(WORK)

if os.path.exists(DEST):
    shutil.rmtree(DEST)
    print('[SETUP] Removed old repo.')

print('Cloning...')
!git clone {REPO} {DEST}

for required in ['train_v3.py', 'evaluate_v3.py', 'models/ecabsd_v3_model.py']:
    if not os.path.exists(os.path.join(DEST, required)):
        raise RuntimeError(f'Clone failed! {required} not found.')

os.chdir(DEST)
print('PWD:', os.getcwd())
print('Repo files:', sorted(os.listdir('.')))
print('\u2705 Repo cloned successfully')

In [ ]:
# ============================================================
# CELL 2: GPU check + install dependencies
# ============================================================
import subprocess, sys, torch

print('[GPU]', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU DETECTED')
print('[PyTorch]', torch.__version__)

def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

print('[DEPS] Installing packages...')
pip_install(['biopython', 'scikit-learn', 'pandas', 'pyyaml'])

print('[DEPS] Installing torch_geometric (may take 2-3 min)...')
pip_install(['torch_geometric'])
pip_install(['pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv'],)

print('[DEPS] \u2705 All dependencies installed.')

In [ ]:
# ============================================================
# CELL 3: Find and link dataset
# ============================================================
import os, shutil

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

input_base = '/kaggle/input'
ds_dir = None

# Auto-scan for dataset folder containing processed/ and splits.csv
for root, dirs, files in os.walk(input_base):
    if 'processed' in dirs and 'splits.csv' in files:
        ds_dir = root
        break

if not ds_dir:
    raise RuntimeError('Dataset not found! Attach the ecabsd-dataset to this notebook.')

print(f'[DATA] Found dataset at: {ds_dir}')

# Copy processed graphs and splits
if os.path.exists('data/processed'):
    shutil.rmtree('data/processed')
os.makedirs('data/processed', exist_ok=True)

shutil.copytree(os.path.join(ds_dir, 'processed'), 'data/processed', dirs_exist_ok=True)
shutil.copy2(os.path.join(ds_dir, 'splits.csv'), 'data/splits.csv')

import glob
print(f'[DATA] Graphs copied: {len(glob.glob("data/processed/*.pt"))}')
print('[DATA] \u2705 Data ready.')

In [ ]:
# ============================================================
# CELL 4: Rebuild splits with zero-overlap (leakage fix)
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/splits.csv')
unique_pdbs = df['pdb_id'].unique()
print(f'[SPLITS] Total unique PDB complexes: {len(unique_pdbs)}')

train_ids, temp_ids = train_test_split(unique_pdbs, test_size=0.3, random_state=42)
val_ids,   test_ids = train_test_split(temp_ids,    test_size=0.5, random_state=42)

df.loc[df['pdb_id'].isin(train_ids), 'split'] = 'train'
df.loc[df['pdb_id'].isin(val_ids),   'split'] = 'val'
df.loc[df['pdb_id'].isin(test_ids),  'split'] = 'test'
df.to_csv('data/splits.csv', index=False)

vc = df['split'].value_counts()
print(f'  train : {vc.get("train", 0)} residues')
print(f'  val   : {vc.get("val",   0)} residues')
print(f'  test  : {vc.get("test",  0)} residues')

# Verify zero overlap
t = set(train_ids); v = set(val_ids); e = set(test_ids)
assert len(t & v) == 0, 'Train-Val overlap!'
assert len(t & e) == 0, 'Train-Test overlap!'
assert len(v & e) == 0, 'Val-Test overlap!'
print('[LEAKAGE] \u2705 Zero overlap confirmed.')

In [ ]:
# ============================================================
# CELL 5: Remove malformed graphs (bad dims)
# ============================================================
import torch, glob, os, shutil

SRC = 'data/processed'
BAD = 'data/bad_graphs'
os.makedirs(BAD, exist_ok=True)

good = bad = 0

for f in glob.glob(SRC + '/*.pt'):
    try:
        g = torch.load(f, map_location='cpu', weights_only=False)
        x_ok = hasattr(g, 'x') and g.x is not None and g.x.dim() == 2 and g.x.shape[1] == 33
        e_ok = hasattr(g, 'edge_attr') and g.edge_attr is not None and g.edge_attr.dim() == 2 and g.edge_attr.shape[1] == 5
        if x_ok and e_ok:
            good += 1
        else:
            bad += 1
            shutil.move(f, os.path.join(BAD, os.path.basename(f)))
    except Exception as ex:
        bad += 1
        shutil.move(f, os.path.join(BAD, os.path.basename(f)))

print(f'Good graphs  : {good}')
print(f'Bad (moved)  : {bad}')
print(f'Final in /processed: {len(glob.glob(SRC + "/*.pt"))}')

if good == 0:
    raise RuntimeError('No valid graphs found. Check dataset.')

print('\u2705 Graph validation done.')

In [ ]:
# ============================================================
# CELL 6: Set V3 config
# ============================================================
import yaml, os, pandas as pd

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# --- Data ---
cfg['data']['processed_dir'] = 'data/processed'
cfg['data']['splits_csv']    = 'data/splits.csv'

# --- Model (V3) ---
cfg['model']['version']          = 'v3'
cfg['model']['esm_dim']          = 33
cfg['model']['edge_feature_dim'] = 5
cfg['model']['hidden_dim']       = 256    # V3: larger capacity
cfg['model']['num_gcn_layers']   = 6      # V3: deeper GATv2
cfg['model']['num_heads']        = 4
cfg['model']['dropout']          = 0.3

# --- Training ---
cfg['training']['epochs']                 = 120
cfg['training']['learning_rate']          = 0.0003
cfg['training']['weight_decay']           = 0.005
cfg['training']['batch_size']             = 8
cfg['training']['loss']                   = 'combined'
cfg['training']['focal_alpha']            = 0.75
cfg['training']['focal_gamma']            = 2.0
cfg['training']['dice_weight']            = 0.5
cfg['training']['warmup_epochs']          = 10
cfg['training']['early_stopping_patience']= 60
cfg['training']['gradient_clip']          = 1.0
cfg['training']['chain_swap_prob']        = 0.5
cfg['training']['num_workers']            = 0
cfg['training']['seed']                   = 42

# --- Paths ---
cfg['paths']['checkpoints_dir'] = 'checkpoints'
cfg['paths']['logs_dir']        = 'logs'
cfg['paths']['results_dir']     = 'results'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Verify
df = pd.read_csv('data/splits.csv')
vc = df['split'].value_counts()
print('=== V3 Config Set ===')
print(f'  hidden_dim    : {cfg["model"]["hidden_dim"]}')
print(f'  num_gcn_layers: {cfg["model"]["num_gcn_layers"]}')
print(f'  epochs        : {cfg["training"]["epochs"]}')
print(f'  warmup        : {cfg["training"]["warmup_epochs"]}')
print(f'  train samples : {vc.get("train", 0)}')
print(f'  val samples   : {vc.get("val",   0)}')
print(f'  test samples  : {vc.get("test",  0)}')
print('\n\u2705 Config ready — safe to train V3!')

In [ ]:
# ============================================================
# CELL 7: TRAIN V3
# ============================================================
import os, sys, subprocess, glob

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

print('[TRAIN] Current dir:', os.getcwd())

# 1. Restore data module (dataset copy might have overwritten it)
print('[TRAIN] Restoring repo data module...')
subprocess.run(['git', 'checkout', '--', 'data/dataset.py', 'data/__init__.py'], cwd=WORKDIR)

if not os.path.exists('data/dataset.py'):
    raise FileNotFoundError('data/dataset.py missing after restore!')

# 2. Sanity check files
print('[TRAIN] data folder:', os.listdir('data')[:20])
print('[TRAIN] Graphs:', len(glob.glob('data/processed/*.pt')))
print('[TRAIN] Splits exists:', os.path.exists('data/splits.csv'))
print('[TRAIN] train_v3.py exists:', os.path.exists('train_v3.py'))
print('[TRAIN] ecabsd_v3_model.py exists:', os.path.exists('models/ecabsd_v3_model.py'))

if len(glob.glob('data/processed/*.pt')) == 0:
    raise RuntimeError('No .pt graphs found in data/processed')

# 3. Run V3 training
print('\n[TRAIN] Starting ECABSD V3 training...')
process = subprocess.Popen(
    [sys.executable, '-u', 'train_v3.py'],
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError('[TRAIN] \u274c Training failed. Check error above.')
else:
    print('[TRAIN] \u2705 V3 training completed successfully!')

In [ ]:
# ============================================================
# CELL 8: Evaluate V3 on test set
# ============================================================
import subprocess, sys, os

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

print('[EVAL] Running evaluate_v3.py on test set...')
process = subprocess.Popen(
    [sys.executable, '-u', 'evaluate_v3.py'],
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    print('[EVAL] \u26a0\ufe0f evaluate_v3.py returned non-zero. Check above.')
else:
    print('[EVAL] \u2705 Evaluation complete.')

In [ ]:
# ============================================================
# CELL 9: Verify checkpoint + package for download
# ============================================================
import os, time, zipfile

WORKDIR = '/kaggle/working/ecabsd'
os.chdir(WORKDIR)

ckpt = 'checkpoints/best_model_v3.pt'

if os.path.exists(ckpt):
    mt = os.path.getmtime(ckpt)
    sz = os.path.getsize(ckpt)
    print('\u2705 Checkpoint verified:')
    print(f'  best_model_v3.pt : {time.ctime(mt)}  ({sz // 1024} KB)')
else:
    print('\u274c ERROR: best_model_v3.pt not found! Training may have failed.')

# Package results
zip_path = '/kaggle/working/ecabsd_v3_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Checkpoints
    for root, _, files in os.walk('checkpoints'):
        for f in files:
            zf.write(os.path.join(root, f))
    # Logs (training history)
    for root, _, files in os.walk('logs'):
        for f in files:
            zf.write(os.path.join(root, f))
    # Results (eval outputs)
    for root, _, files in os.walk('results'):
        for f in files:
            zf.write(os.path.join(root, f))
    # V3 Model code
    for root, _, files in os.walk('models'):
        for f in files:
            if f.endswith('.py'):
                zf.write(os.path.join(root, f))
    # Config
    zf.write('config.yaml')

print(f'\n\u2705 Download ready: {zip_path}')
print('Extract and copy checkpoints/ + logs/ + results/ into your local ecabsd folder.')